# Munti — full training run (free Kaggle T4)

This notebook holds **no logic**. Everything it calls lives in the `munti/`
package in the repo, which is committed and tested — the notebook only wires
up paths and presses go. That keeps the run reproducible (NFR-3) and stops the
notebook and the repo from drifting apart.

## Running it

Either push it with the Kaggle CLI:

```bash
python -m kaggle datasets create -p <staged-repo> -r zip   # once
python -m kaggle kernels push -p <kernel-dir>
python -m kaggle kernels status <user>/munti-train
```

or do it by hand: zip the repo → upload as a Kaggle **Dataset** → New Notebook →
*Add Data* → that dataset → **Accelerator: GPU T4** and **Internet: On** → Run
All → *Save Version → Save & Run All* so `/kaggle/working` persists.

The repo is located by searching the attached datasets for `pyproject.toml`, so
the dataset can be called anything.

## If the session dies (9h limit)

Attach **this notebook's output** alongside the repo dataset. Cell 2 finds the
old checkpoint automatically and training resumes at the exact step — optimizer
and scaler state included, so it's a true continuation, not a warm restart.

In [ ]:
# --- 1. locate the repo, make a writable copy, import the package ---
import os, shutil, subprocess, sys, glob
from pathlib import Path

WORK = Path("/kaggle/working/munti-repo")  # writable copy: data/ and out/ are written here

if not WORK.exists():
    # Find the repo by looking for pyproject.toml anywhere under the attached
    # datasets, rather than hardcoding a slug — the dataset name and any folder
    # nesting inside the zip then both stop mattering.
    roots = [Path(p).parent for p in glob.glob("/kaggle/input/*/**/pyproject.toml", recursive=True)]
    assert roots, "repo not found in /kaggle/input — is the dataset attached?"
    shutil.copytree(roots[0], WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

# A dataset uploaded without -r zip keeps only the top-level files and silently
# drops every directory, so the repo looks fine until an import fails minutes in.
for need in ("munti/model.py", "configs/munti-12m.yaml"):
    assert (WORK / need).exists(), f"{need} missing — re-upload the dataset with: kaggle datasets version -r zip"

import torch

# Preflight. Kaggle hands out whichever accelerator is free, and a P100 (sm_60)
# is *not supported* by the installed PyTorch — every CUDA kernel launch fails.
# The failure would otherwise surface ~10 minutes later, after tokenizing, since
# everything before training is CPU work. Fail here instead.
assert torch.cuda.is_available(), "no GPU — set the accelerator in notebook settings"
cap = torch.cuda.get_device_capability()
print("gpu:", torch.cuda.get_device_name(0), f"sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), (
    f"{torch.cuda.get_device_name(0)} is sm_{cap[0]}{cap[1]}; this PyTorch needs sm_70+. "
    "Push with: kaggle kernels push --accelerator NvidiaTeslaT4"
)
print("bf16 hardware:", cap[0] >= 8, "(Ampere+; T4 is fp16 — the loop handles both)")
print("repo:", WORK, sorted(p.name for p in WORK.iterdir()))

In [ ]:
# --- 2. restore a previous session's checkpoint, if one was attached ---
# Looks through every attached dataset for a prior run's out/ directory.
RESUME = False
for ckpt in glob.glob("/kaggle/input/*/**/out/ckpt.pt", recursive=True):
    prev = Path(ckpt).parent
    shutil.copytree(prev, WORK / "out", dirs_exist_ok=True)
    # The tokenizer must be the one that produced those weights, not a new one.
    tok = Path(ckpt).parent.parent / "data" / "tokenizer.json"
    if tok.exists():
        (WORK / "data").mkdir(exist_ok=True)
        shutil.copy(tok, WORK / "data" / "tokenizer.json")
    RESUME = True
    print("resuming from", ckpt)
    break
else:
    print("fresh run")

In [ ]:
# --- 3. correctness gate ---
# ~40s on CPU, and it has already passed locally. Re-running it here proves the
# code that actually got uploaded is the code that passed, not a stale zip.
print(subprocess.run([sys.executable, "test_munti.py"], capture_output=True, text=True).stdout)

In [ ]:
# --- 4. data: download TinyStories, train the BPE, write the token streams ---
# A few minutes. Skipped if a previous session already built them.
from munti import data as D

if not (WORK / "data" / "train.bin").exists():
    D.prepare(limit=None, vocab_size=4096)   # limit=None => the full 2.1M stories
else:
    print("token streams already present, skipping")
print("train tokens:", f"{len(D.load_split('train')):,}")

In [ ]:
# --- 5. train ---
# ~12.5M params, 20k steps at batch 64 x 256 tokens (~330M tokens seen).
# Checkpoints + probe samples land in out/ every 500 steps.
from munti.train import train

train("configs/munti-12m.yaml", resume=RESUME)

In [ ]:
# --- 6. artifacts: loss curve + generations from held-out prompts ---
from munti.train import plot_curve
from munti.model import Munti
from munti.sample import generate_text
from munti import tokenizer as tk

plot_curve("out/loss.csv")

model = Munti.from_checkpoint(torch.load("out/ckpt.pt", map_location="cuda", weights_only=False), device="cuda")
tok = tk.load()
for prompt in [
    "Once upon a time, there was a little girl named Lily.",
    "Tom found a shiny red box under the tree. He",
    "The cat was hungry, so",
]:
    print("-" * 70)
    print(generate_text(model, tok, prompt, device="cuda", max_new_tokens=200, temperature=0.8, top_k=200))

In [ ]:
# --- 7. things a 12M model should fail at (for the honest can-do/can't-do section) ---
# Not a formality: these outputs go in the README verbatim. TinyStories teaches
# simple narrative prose and nothing else, so questions, facts and dialogue
# should all come back as story-shaped noise.
for prompt in [
    "What is the capital of France?",
    "Q: How many legs does a spider have? A:",
    "def fibonacci(n):",
    "The mitochondria is",
]:
    print("-" * 70)
    print(generate_text(model, tok, prompt, device="cuda", max_new_tokens=80, temperature=0.8, top_k=200))

In [ ]:
# --- 8. bundle the artifacts to download ---
# Commit these to the repo: loss.csv, curve.png, samples.md, tokenizer.json
# (force-add the tokenizer — it's gitignored so a dev-slice one can't sneak in).
shutil.copy("data/tokenizer.json", "out/tokenizer.json")
shutil.make_archive("/kaggle/working/munti-artifacts", "zip", "out")
print(sorted(os.listdir("out")))